# Genome-Doc NRE Training — Colab

Trains the **Neural Re-Rendering Engine (NRE)** with text conditioning fix.

**Prerequisites:**
- Upload `genome-doc/` project folder to Google Drive
- Upload SIR checkpoint to Google Drive
- Use GPU runtime: `Runtime > Change runtime type > GPU`

## 1. Check GPU & Mount Drive

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)')
else:
    raise RuntimeError('No GPU! Go to Runtime > Change runtime type > GPU')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Configure Paths
Edit these to match your Google Drive layout.

In [ ]:
import os

# === EDIT THESE PATHS ===
PROJECT_DIR = '/content/drive/MyDrive/prometheus/genome-doc'
SIR_CHECKPOINT = '/content/drive/MyDrive/prometheus/checkpoints/sir/best_model.pt'
TRAIN_DATA_DIR = f'{PROJECT_DIR}/data/synthetic/train'
VAL_DATA_DIR = f'{PROJECT_DIR}/data/synthetic/val'
OUTPUT_DIR = '/content/drive/MyDrive/prometheus/checkpoints/nre'
# ========================

for name, path in [('Project', PROJECT_DIR), ('Train data', TRAIN_DATA_DIR)]:
    status = 'OK' if os.path.exists(path) else 'NOT FOUND'
    print(f'[{status}] {name}: {path}')

sir_ok = os.path.exists(SIR_CHECKPOINT)
print(f"[{'OK' if sir_ok else 'WARN'}] SIR checkpoint: {SIR_CHECKPOINT}")
if not sir_ok:
    print('  -> Will use random style embeddings (testing only)')

if not os.path.exists(VAL_DATA_DIR):
    print('Val data not found, using train data for validation')
    VAL_DATA_DIR = TRAIN_DATA_DIR

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'\nCheckpoints -> {OUTPUT_DIR}')

## 3. Install Dependencies

In [ ]:
!pip install -q diffusers>=0.25.0 transformers>=4.36.0 accelerate>=0.25.0 peft>=0.7.0 safetensors>=0.4.0 albumentations>=1.3.0 pydantic>=2.5.0 pyyaml>=6.0 torchmetrics>=1.2.0 lpips>=0.1.4 timm>=0.9.0 wandb>=0.16.0 datasets>=2.16.0
print('Dependencies installed')

## 4. Import Project Modules

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

from models.nre.controlnet_renderer import NREControlNet
from models.sir.style_encoder import StyleEncoder
from data.dataset import GenomeDocDataset, create_dataloader
from training.losses import RenderingLoss
print('All project modules imported')

## 5. Auto-Configure for GPU

In [ ]:
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

if gpu_mem_gb >= 70:
    BATCH_SIZE, GRAD_ACCUM, NUM_WORKERS = 8, 4, 4
elif gpu_mem_gb >= 38:
    BATCH_SIZE, GRAD_ACCUM, NUM_WORKERS = 4, 8, 4
elif gpu_mem_gb >= 14:
    BATCH_SIZE, GRAD_ACCUM, NUM_WORKERS = 2, 16, 2
else:
    BATCH_SIZE, GRAD_ACCUM, NUM_WORKERS = 1, 32, 2

config = {
    'model': {'nre': {
        'sd_model_id': 'stable-diffusion-v1-5/stable-diffusion-v1-5',
        'controlnet_model_id': 'lllyasviel/sd-controlnet-canny',
        'style_dim': 512, 'num_style_tokens': 4,
        'lora_rank': 8, 'lora_alpha': 16, 'use_lora': True,
    }},
    'training': {
        'batch_size': BATCH_SIZE, 'gradient_accumulation': GRAD_ACCUM,
        'max_epochs': 30, 'learning_rate': 1e-5, 'weight_decay': 0.01,
        'fp16': True, 'gradient_checkpointing': True,
        'gradient_clip': 1.0, 'patience': 15,
        'loss_weights': {'l1': 1.0, 'lpips': 0.5, 'style': 0.1},
    },
    'data': {'image_size': [512, 512], 'num_workers': NUM_WORKERS},
    'logging': {'log_every_n_steps': 25, 'save_every_n_epochs': 5},
}

print(f'GPU: {torch.cuda.get_device_name(0)} ({gpu_mem_gb:.0f} GB)')
print(f'Batch: {BATCH_SIZE} x {GRAD_ACCUM} accum = {BATCH_SIZE * GRAD_ACCUM} effective')
print(f'Epochs: {config["training"]["max_epochs"]}, LR: {config["training"]["learning_rate"]}')

## 6. Load Data

In [ ]:
print('Loading training data...')
train_loader = create_dataloader(
    data_dir=TRAIN_DATA_DIR, mode='nre', batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS,
    image_size=tuple(config['data']['image_size']),
)
print(f'  {len(train_loader.dataset)} train samples, {len(train_loader)} batches/epoch')

print('Loading validation data...')
val_loader = create_dataloader(
    data_dir=VAL_DATA_DIR, mode='nre', batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS,
    image_size=tuple(config['data']['image_size']),
)
print(f'  {len(val_loader.dataset)} val samples')

# Sanity check
sample = next(iter(train_loader))
print(f'\nBatch keys: {list(sample.keys())}')
for k, v in sample.items():
    if isinstance(v, torch.Tensor):
        print(f'  {k}: {v.shape}')
    elif isinstance(v, list) and v and isinstance(v[0], str):
        print(f'  {k}: list[str] len={len(v)}, sample={v[0][:80]}...')

## 7. Initialize & Train

In [ ]:
from training.train_nre import NRETrainer

trainer = NRETrainer(
    config=config,
    sir_checkpoint=SIR_CHECKPOINT if os.path.exists(SIR_CHECKPOINT) else None,
    device='auto',
)
print(f'Trainer ready. {trainer._vram_stats()}')

In [ ]:
# START TRAINING (checkpoints save to Google Drive)
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    output_dir=OUTPUT_DIR,
    resume_from=None,  # Set path to resume after disconnect
)

## 8. Resume After Disconnect
If Colab disconnects, re-run cells 1-6, then this cell.

In [ ]:
# Uncomment to resume:
# import glob
# checkpoints = sorted(glob.glob(f'{OUTPUT_DIR}/checkpoint_epoch_*.pt'))
# if checkpoints:
#     print(f'Resuming from: {checkpoints[-1]}')
#     trainer.train(train_loader, val_loader, OUTPUT_DIR, resume_from=checkpoints[-1])
# else:
#     print('No checkpoints found')

## 9. Inference Test

In [ ]:
import matplotlib.pyplot as plt

test_batch = next(iter(val_loader))
skeleton = test_batch['skeleton'][:2].to(trainer.device)
style_patches = test_batch['style_patches'][:2].to(trainer.device)
text_prompt = test_batch['token_sequence'][:2]
clean_gt = test_batch['clean_image'][:2]

style_emb = trainer._extract_style(style_patches)

trainer.model.eval()
with torch.no_grad():
    generated = trainer.model.generate(
        skeleton=skeleton, style_embedding=style_emb,
        text_prompt=text_prompt, num_inference_steps=30, guidance_scale=7.5,
    )

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
titles = ['Skeleton', 'Generated (NRE)', 'Ground Truth']
for i in range(2):
    imgs = [skeleton[i], generated[i], clean_gt[i]]
    for j, (img, title) in enumerate(zip(imgs, titles)):
        axes[i, j].imshow(img.cpu().permute(1, 2, 0).clamp(0, 1))
        axes[i, j].set_title(title)
        axes[i, j].axis('off')

plt.suptitle('NRE Inference — Text Hallucination Fix', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/inference_test.png', dpi=150, bbox_inches='tight')
plt.show()

for i, t in enumerate(text_prompt):
    print(f'[{i}] {t[:120]}...')